In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import scipy

base_size = 12

plt.rcParams.update({

    'font.size': base_size,           
    'axes.titlesize': base_size + 2,   
    'axes.labelsize': base_size,       
    'xtick.labelsize': base_size - 2,  
    'ytick.labelsize': base_size - 2,  
    'legend.fontsize': base_size - 1,  
    'figure.titlesize': base_size + 4, 
    
    'figure.figsize': (10, 5),         
    'figure.dpi': 100,                 
    'savefig.dpi': 300,                
    'axes.grid': True,                 
    'grid.alpha': 0.3,                 
    'grid.linestyle': '--'             
})

## Task 1

In [ ]:
np.random.seed(42)  # for reproducibility
P = np.array([
    [0.9915, 0.0050, 0.0025, 0.0000, 0.0010],
    [0.0000, 0.9860, 0.0050, 0.0040, 0.0050],
    [0.0000, 0.0000, 0.9920, 0.0030, 0.0050],
    [0.0000, 0.0000, 0.0000, 0.9910, 0.0090],
    [0.0000, 0.0000, 0.0000, 0.0000, 1.0000],
], dtype=float)

n_women = 1000
X0 = 0        
n_states = P.shape[0]
DEAD = n_states - 1  

def sim_cancer_markov_until_all_dead(P, n_women, X0=0):
    current = np.full(n_women, X0, dtype=int)
    history = [current.copy()]

    while not np.all(current == DEAD):
        next_state = np.array([
            np.random.choice(P.shape[0], p=P[current[w]])
            for w in range(n_women)
        ])
        # Once dead, stay dead
        current = np.where(current == DEAD, DEAD, next_state)
        history.append(current.copy())

    return np.array(history).T  # shape: (n_women, T+1)

states_over_time = sim_cancer_markov_until_all_dead(P, n_women, X0)
T = states_over_time.shape[1] - 1
print(f"All {n_women} women reached Dead state after {T} time steps.")


In [ ]:
np.random.seed(42)  
state_labels = ["Surgery", "Local recurrence", "Distant metastasis", "Both", "Dead"]

# Simulated proportions: fraction of women in each state at each time step
proportions = np.zeros((T + 1, n_states))
for t in range(T + 1):
    for s in range(n_states):
        proportions[t, s] = np.sum(states_over_time[:, t] == s) / n_women

# Theoretical proportions via matrix power: pi(t) = pi(0) @ P^t (from project beskrivelse)
pi = np.zeros((T + 1, n_states))
pi[0, X0] = 1.0
Pt = np.eye(n_states)
for t in range(1, T + 1):
    Pt = Pt @ P
    pi[t] = pi[0] @ Pt

time = np.arange(T + 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for s in range(n_states):
    axes[0].plot(time, proportions[:, s], label=state_labels[s])
axes[0].set_title(f"Simulated state proportions over time\n({n_women} women, all until Dead)")
axes[0].set_xlabel("Time (months)")
axes[0].set_ylabel("Proportion of women")
axes[0].legend(loc="center right")
axes[0].grid(True, alpha=0.3)

for s in range(n_states):
    axes[1].plot(time, pi[:, s], linestyle="--", label=state_labels[s])
axes[1].set_title("Theoretical state proportions over time")
axes[1].set_xlabel("Time (months)")
axes[1].set_ylabel("Proportion of women")
axes[1].legend(loc="center right")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


In [ ]:
# Proportion of women who ever enter state 1 (local recurrence)
ever_local = np.any((states_over_time == 1) | (states_over_time == 3), axis=1)
prob_local_recurrence = ever_local.mean() * 100
print(f"Proportion of women where cancer eventually reappears locally: {prob_local_recurrence:.2f}%")


## Task 2

In [ ]:
#distributions over states at t = 120
dist_120 = states_over_time[:, 120]
state_props_120 = np.bincount(dist_120, minlength=n_states) / n_women
colors = plt.cm.Set2(np.linspace(0, 1, n_states))

dist_120_ex = pi[120]  # Theoretical distribution at t=120 from matrix power
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Simulated
bars_sim = axes[0].bar(
    range(n_states),
    state_props_120,
    tick_label=state_labels,
    color=colors,
    edgecolor="black",
    linewidth=0.8
)
axes[0].set_title("Simulated Distribution at t = 120", pad=10)
axes[0].set_xlabel("State")
axes[0].set_ylabel("Proportion")
axes[0].set_ylim(0, 1)
axes[0].grid(axis="y", linestyle="--", alpha=0.3)
for bar, v in zip(bars_sim, state_props_120):
    axes[0].text(bar.get_x() + bar.get_width() / 2, v + 0.01, f"{v:.1%}",
                 ha="center", va="bottom", fontsize=9)

# Theoretical
bars_th = axes[1].bar(
    range(n_states),
    dist_120_ex,
    tick_label=state_labels,
    color=colors,
    edgecolor="black",
    linewidth=0.8
)
axes[1].set_title("Theoretical Distribution at t = 120", pad=10)
axes[1].set_xlabel("State")
axes[1].set_ylabel("Proportion")
axes[1].set_ylim(0, 1)
axes[1].grid(axis="y", linestyle="--", alpha=0.3)
for bar, v in zip(bars_th, dist_120_ex):
    axes[1].text(bar.get_x() + bar.get_width() / 2, v + 0.01, f"{v:.1%}",
                 ha="center", va="bottom", fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
#chi squared test for distribution at t=120
count_expected = state_props_120 * n_women
count_observed = np.bincount(dist_120, minlength=n_states)

# Recompute test as 1D observed vs 1D expected counts
count_expected = dist_120_ex * n_women
count_expected *= count_observed.sum() / count_expected.sum()  # ensure equal totals

chi2, p = scipy.stats.chisquare(f_obs=count_observed, f_exp=count_expected)
print(f"Chi-squared statistic: {chi2:.4f}, p-value: {p:.6f}")

## Task 3

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.stats as stats
from numpy.linalg import inv, matrix_power

P = np.array([
    [0.9915, 0.005,  0.0025, 0,      0.001 ],
    [0,      0.986,  0.005,  0.004,  0.005 ],
    [0,      0,      0.992,  0.003,  0.005 ],
    [0,      0,      0,      0.991,  0.009 ],
    [0,      0,      0,      0,      1.0   ]
])

def simulate_woman(P):
    """Simulates a single woman's path until death (state 5, index 4)."""
    state = 0
    path = [state]
    while state != 4:
        state = np.random.choice(5, p=P[state])
        path.append(state)
    return path

np.random.seed(42)
n_sims = 1000
lifetimes = []
local_recurrence_count = 0
states_at_120 = []

for _ in range(n_sims):
    path = simulate_woman(P)
    lifetimes.append(len(path) - 1)
    
    if 1 in path or 3 in path:
        local_recurrence_count += 1
        
    # State at t = 120
    if len(path) - 1 >= 120:
        states_at_120.append(path[120])
    else:
        states_at_120.append(4)


Ps = P[:4, :4]
ps = P[:4, 4]
pi_s = np.array([1.0, 0.0, 0.0, 0.0])
I = np.eye(4)
N = inv(I - Ps)
e = np.ones(4)


mean_theory = pi_s @ N @ e

t_max = max(lifetimes)
t_vals = np.arange(1, t_max + 1)
pmf_theory = np.array([pi_s @ matrix_power(Ps, t-1) @ ps for t in t_vals])

mean_sim = np.mean(lifetimes)
std_sim = np.std(lifetimes)


bin_edges = np.linspace(1, t_max, 16)
obs_counts, _ = np.histogram(lifetimes, bins=bin_edges)

exp_counts = []
for i in range(len(bin_edges) - 1):
    low, high = int(np.floor(bin_edges[i])), int(np.floor(bin_edges[i+1]))

    prob = np.sum(pmf_theory[(t_vals >= low) & (t_vals < high)])
    exp_counts.append(prob * n_sims)

exp_counts = np.array(exp_counts)

exp_counts = exp_counts * (np.sum(obs_counts) / np.sum(exp_counts))

chi2_stat_fit, p_val_fit = stats.chisquare(f_obs=obs_counts, f_exp=exp_counts)


print("="*60)
print(f"{'STATISTIC':<30} | {'SIMULATED':<12} | {'THEREOTICAL':<12}")
print("="*60)
print(f"{'Mean Lifetime (Months)':<30} | {mean_sim:<12.2f} | {mean_theory:<12.2f}")

print("-"*60)
print(f"Global Chi-Square Goodness-of-Fit Test (15 bins):")
print(f"  - Chi2 Statistic: {chi2_stat_fit:.4f}")
print(f"  - p-value: {p_val_fit:.4f} (A high p-value means the distributions match perfectly!)")
print("="*60)

bin_labels = [f"{int(bin_edges[i])}-{int(bin_edges[i+1])}" for i in range(len(bin_edges)-2)] + [f"{int(bin_edges[-2])}+"]
x = np.arange(len(bin_labels))
width = 0.42

fig, ax = plt.subplots()

observed = obs_counts
expected = exp_counts

ax.bar(x - width/2, observed, width=width, color="#4C78A8", edgecolor="black", label="Observed")
ax.bar(x + width/2, expected, width=width, color="#F58518", edgecolor="black", alpha=0.85, label="Expected (theory)")

ax.set_title("Lifetimes", pad=10)
ax.set_xlabel("Months")
ax.set_ylabel("Number of women")
ax.set_xticks(x)
ax.set_xticklabels(bin_labels, rotation=45, ha="center")
ax.legend()

plt.tight_layout()
plt.show()

## Task 4

In [ ]:
np.random.seed(42)

accepted_lifetimes = []
while len(accepted_lifetimes) < 1000:
    path = simulate_woman(P)
    
    # 1. Condition: Survived at least 12 months
    survived_12 = (len(path) - 1 >= 12)
    
    # 2. Condition: Cancer reappeared at any point in months 1 to 12 
    early_recurrence = any(state in [1, 2, 3] for state in path[1:13])
    
    if survived_12 and early_recurrence:
        accepted_lifetimes.append(len(path) - 1)

print(f"Task 4: Expected lifetime of a woman with early recurrence: {np.mean(accepted_lifetimes):.2f} months")

## Task 5

In [ ]:
np.random.seed(42)

n_iters = 100
n_women = 200

Y_vals, X_vals = [], []

for _ in range(n_iters):
    batch_lifetimes = [len(simulate_woman(P)) - 1 for _ in range(n_women)]
    # Y = fraction dying within 350 months (our target estimator)
    Y_vals.append(np.mean(np.array(batch_lifetimes) <= 350))
    # X = mean lifetime of the batch (our control variate)
    X_vals.append(np.mean(batch_lifetimes))

Y_vals = np.array(Y_vals)
X_vals = np.array(X_vals)

cov_matrix = np.cov(X_vals, Y_vals)
c_star = -cov_matrix[0, 1] / cov_matrix[0, 0]

Y_c = Y_vals + c_star * (X_vals - mean_theory)

var_crude = np.var(Y_vals, ddof=1)
var_cv = np.var(Y_c, ddof=1)
reduction = (1 - (var_cv / var_crude)) * 100

print(f"Task 5: Crude MC estimate P(T <= 350): {np.mean(Y_vals):.4f} (Var: {var_crude:.6f})")
print(f"Task 5: Control Variate estimate:      {np.mean(Y_c):.4f} (Var: {var_cv:.6f})")
print(f"Task 5: Variance reduction achieved:   {reduction:.2f}%")

## Task 7

In [ ]:
Q = np.array([[-0.0085, 0.005, 0.0025, 0, 0.001],
              [0, -0.014, 0.005, 0.004, 0.005],
              [0, 0, -0.008, 0.003, 0.005],
              [0, 0, 0, -0.009, 0.009],
              [0, 0, 0, 0, 0]])
n_women = 1000
X0 = 0
n_states = Q.shape[0]
DEAD = n_states - 1  

def simulate_ctmc(Q, n_women, X0=0):
    lifetimes = np.zeros(n_women)
    distant_after_30_5 = np.zeros(n_women, dtype=bool)

    for w in range(n_women):
        state = X0
        time = 0.0

        while state != DEAD:
            rate = -Q[state, state]

            waiting_time = np.random.exponential(scale=1 / rate)
            time += waiting_time

            probs = Q[state].copy()
            probs[state] = 0
            probs = probs / rate

            next_state = np.random.choice(Q.shape[0], p=probs)

            if time > 30.5 and next_state in [2, 3]:
                distant_after_30_5[w] = True

            state = next_state

        lifetimes[w] = time

    return lifetimes, distant_after_30_5

lifetimes, distant_after_30_5 = simulate_ctmc(Q, n_women, X0)

In [ ]:
distant_after_30_5_proportion = distant_after_30_5.mean() * 100
print(f"Proportion of women who develop distant metastasis after 30.5 months: {distant_after_30_5_proportion:.2f}%")

In [ ]:
from scipy.stats import t, chi2

mean_lifetime = np.mean(lifetimes)
sd_lifetime = np.std(lifetimes, ddof=1)

alpha = 0.05
n = len(lifetimes)

# CI for mean
se = sd_lifetime / np.sqrt(n)
t_crit = t.ppf(1 - alpha / 2, df=n - 1)

mean_ci = (
    mean_lifetime - t_crit * se,
    mean_lifetime + t_crit * se
)

# CI for standard deviation
sd_ci = (
    np.sqrt((n - 1) * sd_lifetime**2 / chi2.ppf(1 - alpha / 2, df=n - 1)),
    np.sqrt((n - 1) * sd_lifetime**2 / chi2.ppf(alpha / 2, df=n - 1))
)

print(f"Mean lifetime: {mean_lifetime:.2f} months")
print(f"95% CI for mean: ({mean_ci[0]:.2f}, {mean_ci[1]:.2f})")

print(f"Standard deviation: {sd_lifetime:.2f} months")
print(f"95% CI for SD: ({sd_ci[0]:.2f}, {sd_ci[1]:.2f})")

## Task 8

In [ ]:
Qs = Q[:-1, :-1]
pi0 = np.array([1, 0, 0, 0])  # start in state 0
ones = np.ones(Qs.shape[0])
def theoretical_cdf(t):
    t = np.asarray(t)

    if t.ndim == 0:
        return 1 - pi0 @ scipy.linalg.expm(Qs * t) @ ones

    return np.array([
        1 - pi0 @ scipy.linalg.expm(Qs * ti) @ ones
        for ti in t
    ])

ks_stat, p_value = scipy.stats.kstest(
    lifetimes,
    theoretical_cdf
)

print(f"KS statistic = {ks_stat:.4f}")
print(f"p-value = {p_value:.4f}")

In [ ]:
#plotting the empirical CDF vs theoretical CDF
t_values = np.linspace(0, lifetimes.max(), 100)
theory_cdf_values = theoretical_cdf(t_values)  
empirical_cdf_values = np.array([
    np.mean(lifetimes <= t) for t in t_values
])

plt.figure()
plt.plot(t_values, empirical_cdf_values, label="Empirical CDF", color="#4C78A8")
plt.plot(t_values, theory_cdf_values, label="Theoretical CDF", color="#F58518", linestyle="--")
plt.title("Empirical vs Theoretical CDF of Lifetimes")
plt.xlabel("Time (months)")
plt.ylabel("Cumulative Probability")
plt.legend()
plt.tight_layout()
plt.show()

## Task 9

In [ ]:
# Task 7

import numpy as np
import math
import matplotlib.pyplot as plt
import scipy.stats as stats


Q = np.array([
    [-0.0085, 0.005,  0.0025, 0,      0.001],
    [0,      -0.014,  0.005,  0.004,  0.005],
    [0,       0,     -0.008,  0.003,  0.005],
    [0,       0,      0,     -0.009,  0.009],
    
])

np.random.seed(42)
n_women = 1000
lifetimes = []
distant_recurrence_count = 0

states_list = [0,1,2,3,4] 

state_120 = []

for _ in range(n_women):
    current_state = 0 
    time = 0
    had_distant_recurrence = False
    
    while current_state != 4:
        rate = -Q[current_state, current_state]
        sojourn_time = np.random.exponential(scale = 1/rate)
        time += sojourn_time


        transition_prob = Q[current_state].copy()
        transition_prob[current_state] = 0
        transition_prob = transition_prob/rate

        current_state = np.random.choice(states_list, p = transition_prob)
        
        if current_state == 2 or current_state == 3:
            if time > 30.5:
                had_distant_recurrence = True
        
            
    lifetimes.append(time)
    
    if had_distant_recurrence:
        distant_recurrence_count += 1


lifetimes = np.array(lifetimes)
mean_lifetime = np.mean(lifetimes)
std_lifetime = np.std(lifetimes, ddof=1)
alpha = 0.05
df = n_women - 1


prop_women = distant_recurrence_count/n_women


import numpy as np
import math
import matplotlib.pyplot as plt
import scipy.stats as stats

Q_new = np.array([
    [-0.00475, 0.0025,  0.00125, 0,      0.001],
    [0,      -0.007,  0,  0.002,  0.005],
    [0,       0,     -0.008,  0.003,  0.005],
    [0,       0,      0,     -0.009,  0.009],
    [0,       0,       0,      0,     0]
])


np.random.seed(42)
n_women = 1000
lifetimes_new = []
distant_recurrence_count_new = 0

states_list = [0,1,2,3,4] 

state_120 = []

for _ in range(n_women):
    current_state = 0 
    time = 0
    had_distant_recurrence = False
    
    while current_state != 4:
        rate = -Q_new[current_state, current_state]
        sojourn_time = np.random.exponential(scale = 1/rate)
        time += sojourn_time


        transition_prob = Q_new[current_state].copy()
        transition_prob[current_state] = 0
        transition_prob = transition_prob/rate

        current_state = np.random.choice(states_list, p = transition_prob)
        
        if current_state == 2 or current_state == 3:
            if time > 30.5:
                had_distant_recurrence = True
        
            
    lifetimes_new.append(time)
    
    if had_distant_recurrence:
        distant_recurrence_count_new += 1


lifetimes_new = np.array(lifetimes_new)

mean_no_treat = np.mean(lifetimes)
mean_treat = np.mean(lifetimes_new)

print(f'mean with no treatment: {mean_no_treat}')
print(f'mean with treatment: {mean_treat}')


def plot_kaplan_meier(lifetimes, label, color):
    sorted_lifetimes = np.sort(lifetimes)
    proportions_alive = 1 - np.arange(1, len(lifetimes) + 1) / len(lifetimes)
    
    plt.step(sorted_lifetimes, proportions_alive, where='post', label=label, color=color, linewidth=2)

plt.figure(figsize=(10, 6))

# Plot both curves
plot_kaplan_meier(lifetimes, label='Standard (No Treatment)', color='red')
plot_kaplan_meier(lifetimes_new, label='Preventive Treatment', color='blue')

plt.title('Kaplan-Meier Survival Curves')
plt.xlabel('Months after Surgery')
plt.ylabel('Proportion Alive $\hat{S}(t)$')
plt.ylim([0, 1.05])
plt.legend()
plt.grid(alpha=0.3)
plt.show()


## Task 12

In [ ]:
# Task 12

import numpy as np
import math
import matplotlib.pyplot as plt
import scipy.stats as stats

Q_new = np.array([
    [-0.00475, 0.0025,  0.00125, 0,      0.001],
    [0,      -0.007,  0,  0.002,  0.005],
    [0,       0,     -0.008,  0.003,  0.005],
    [0,       0,      0,     -0.009,  0.009],
    [0,       0,       0,      0,     0]
])


np.random.seed(42)
n_women = 1000
lifetimes_new = []
monitoring = []

states_list = [0,1,2,3,4] 


for _ in range(n_women):
    current_state = 0 
    time = 0
    monitoring_woman = []
    next_checkup = 0
    
    while current_state != 4:
        rate = -Q_new[current_state, current_state]
        sojourn_time = np.random.exponential(scale = 1/rate)

        while time + sojourn_time >= next_checkup:
            monitoring_woman.append(current_state)
            next_checkup += 48

        time += sojourn_time

        transition_prob = Q_new[current_state].copy()
        transition_prob[current_state] = 0
        transition_prob = transition_prob/rate


        current_state = np.random.choice(states_list, p = transition_prob)

        
    monitoring_woman.append(4)

    lifetimes_new.append(time)
    monitoring.append(monitoring_woman)

Y = monitoring
    

print("Example observations for the first 7 women:")
for i in range(7):
    print(f"Woman {i+1}: {monitoring[i]}")

## Task 13

In [ ]:
# Task 13


def find_Q(Q_initial, data):
    converged = False
    Q_current = Q_initial.copy()
    states_list = [0, 1, 2, 3, 4]

    while not converged:
        N_ij = np.zeros((5, 5))
        S_i = np.zeros(5)

        for woman_obs in data:
            for step in range(len(woman_obs) - 1):
                start_state = woman_obs[step]
                end_state = woman_obs[step + 1]

                accepted = False

                while not accepted:
                    current_state = start_state
                    time = 0

                    temp_N_ij = np.zeros((5, 5))
                    temp_S_i = np.zeros(5)

                    while time < 48:
                        rate = -Q_current[current_state, current_state]

                        if current_state == 4:
                            temp_S_i[current_state] += 48 - time
                            time = 48
                            break

                        sojourn_time = np.random.exponential(scale=1/rate)

                        if time + sojourn_time >= 48:
                            temp_S_i[current_state] += 48 - time
                            time = 48
                            break
                        else:
                            temp_S_i[current_state] += sojourn_time
                            time += sojourn_time
                            
                            transition_probs = Q_current[current_state].copy()
                            transition_probs[current_state] = 0
                            transition_probs = transition_probs / rate
                            
                            next_state = np.random.choice(states_list, p=transition_probs)
                            temp_N_ij[current_state, next_state] += 1
                            current_state = next_state
                    
                    if current_state == end_state:
                        accepted = True
                        N_ij += temp_N_ij
                        S_i += temp_S_i

        Q_next = np.zeros((5, 5))
        for i in range(4):
            for j in range(5):
                if i != j:
                    Q_next[i, j] = N_ij[i, j] / S_i[i] if S_i[i] > 0 else 0
        
            Q_next[i, i] = -np.sum(Q_next[i])
            
        max_diff = np.max(np.abs(Q_current - Q_next))
        print(f"Max difference this iteration: {max_diff:.5f}")
        
        if max_diff < 1e-3:
            converged = True
        
        Q_current = np.copy(Q_next)
    
    return Q_current


Q_curr = find_Q(Q_new, Y)

print("Estimated Q Matrix:")
print(np.round(Q_curr, 4))
print(np.round(Q_new, 4))


